# Step 3 – AUC boxplot

Reads the `tst_part*.csv` files from Step 2 (2 seeds x 5 folds = 10 AUC values per model and ROI) and draws the "All Patients Classification Performance" boxplot. Also writes a summary table.

Run after Step 2. Set the sample paths in the first code cell to your own data locations before running.

In [ ]:
from glob import glob
from math import sqrt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as stats
import seaborn as sns

# ---- User input ----
RESULTS_DIR = "/path/to/BMI_DATA/results"   # per-fold results from Step 2   (SAMPLE PATH)
OUTPUT_DIR = "/path/to/BMI_DATA/output"      # plot and summary are saved here (SAMPLE PATH)
GROUP = "large_group"
MAGNITUDE = "20x"
ROIS = ["WSI", "Tumor", "TumorDense", "TumorNotDense", "NonTumor", "StromaDense", "StromaNotDense"]
# --------------------

ROI_DISPLAY_NAMES = {
    "WSI": "combined tumor/stroma",
    "Tumor": "tumor",
    "NonTumor": "stroma",
    "TumorDense": "tumor HD",
    "TumorNotDense": "tumor LD",
    "StromaDense": "stroma HD",
    "StromaNotDense": "stroma LD",
}

## Load the per-fold results

In [ ]:
frames = []
for roi in ROIS:
    files = glob(f"{RESULTS_DIR}/{MAGNITUDE}/{GROUP}_s*_{roi}/*.csv")
    if not files:
        print(f"Warning: no results found for {roi}")
        continue
    frames.append(pd.concat([pd.read_csv(f).assign(ROI=roi) for f in files], ignore_index=True))
    print(f"Loaded {len(files)} files for {roi}")

data = pd.concat(frames, ignore_index=True)
data["ROI_display"] = data["ROI"].map(ROI_DISPLAY_NAMES)
data.head()

## Boxplot

In [ ]:
out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

sns.set(style="whitegrid", font_scale=1.5)
plt.figure(figsize=(20, 13))
sns.boxplot(data=data, x="method", y="auc", hue="ROI_display", palette="Set2")
sns.stripplot(data=data, x="method", y="auc", hue="ROI_display", dodge=True, size=4, alpha=0.5, legend=False)
plt.title("All Patients Classification Performance", fontsize=30)
plt.ylabel("AUC Score", fontsize=25)
plt.xlabel("Method", fontsize=25)
plt.xticks(rotation=45, fontsize=25)
plt.yticks(fontsize=25)
plt.ylim(0.0, 1.0)
plt.legend(title="ROI", loc="lower right", fontsize=25, title_fontsize=25)
plt.tight_layout()

plot_path = out_dir / f"boxplot_auc_{GROUP}_all_ROI_comparison.png"
plt.savefig(plot_path, dpi=300, bbox_inches="tight")
plt.show()
print("Saved", plot_path)

## Summary table

Mean AUC with 95% CI over the 10 folds, plus accuracy, sensitivity, specificity, F1 and MCC computed from the confusion counts pooled over the 10 folds.

In [ ]:
def pooled_metrics(tp, tn, fp, fn):
    total = tp + tn + fp + fn
    sen = tp / (tp + fn) if (tp + fn) else 0.0
    spe = tn / (tn + fp) if (tn + fp) else 0.0
    pre = tp / (tp + fp) if (tp + fp) else 0.0
    f1 = 2 * pre * sen / (pre + sen) if (pre + sen) else 0.0
    den = sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    mcc = ((tp * tn) - (fp * fn)) / den if den else 0.0
    return {"acc": (tp + tn) / total if total else 0.0, "sen": sen, "spe": spe, "f1": f1, "mcc": mcc}


rows = []
for roi in ROIS:
    for method, g in data[data["ROI"] == roi].groupby("method"):
        auc = g["auc"].values
        ci = stats.t.interval(0.95, df=len(auc) - 1, loc=np.mean(auc), scale=stats.sem(auc))
        tn, fp, fn, tp = g["c00"].sum(), g["c01"].sum(), g["c10"].sum(), g["c11"].sum()
        rows.append({"ROI": roi, "method": method, "auc_avg": np.mean(auc),
                     "auc_CI_lower": ci[0], "auc_CI_upper": ci[1],
                     **pooled_metrics(tp, tn, fp, fn), "tp": tp, "tn": tn, "fp": fp, "fn": fn})

summary = pd.DataFrame(rows)
summary_path = out_dir / f"summary_all_ROIs_{GROUP}.csv"
summary.to_csv(summary_path, index=False)
print("Saved", summary_path)
summary